In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns
from collections import Counter
from itertools import combinations, permutations

1.a

In [ ]:
def configuration_model(degrees, seed=None):
    """Configuration model from a degree sequence (list of integers). Gives node
    i k_i stubs (half-edges), shuffles all the stubs, and pairs them up two at a
    time. Returns an nx.MultiGraph, which may contain self-loops and duplicate
    edges. A self-loop counts twice toward a node's degree, so every node's
    degree equals its entry in the input exactly.
    seed (optional int) makes the shuffle, and so the graph, reproducible.
    Raises ValueError on negative / non-integer degrees or an odd sum.
    Based on the class 04 notebook.
    """
    if any(not isinstance(k, (int, np.integer)) or k < 0 for k in degrees):
        raise ValueError("Degrees must be non-negative integers.")
    if sum(degrees) % 2 != 0:
        raise ValueError("The sum of the degree sequence must be even.")
    rng = np.random.default_rng(seed)
    # node i appears k_i times in the stub list
    stubs = np.repeat(np.arange(len(degrees)), degrees)
    rng.shuffle(stubs)
    G = nx.MultiGraph()
    G.add_nodes_from(range(len(degrees)))
    # consecutive pairs of the shuffled stubs are the edges
    G.add_edges_from(stubs.reshape(-1, 2).tolist())
    return G

Let's test it on the degree sequence from part (c), with `seed=10` for reproducibility. Self-loops count twice toward a node's degree in a `MultiGraph`, so the output degrees should match the input exactly even if the graph has loops or parallel edges. The same seed should give the same graph twice, and a different seed should (almost always) give a different one.

In [ ]:
degrees = [8, 5, 5, 4, 3, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1]
G = configuration_model(degrees, seed=10)
print(type(G).__name__, G.number_of_nodes(), "nodes", G.number_of_edges(), "edges")
print("input: ", degrees)
print("output:", [G.degree(i) for i in range(len(degrees))])
print("degrees match:", [G.degree(i) for i in range(len(degrees))] == degrees)
print("self-loops:", list(nx.selfloop_edges(G)))
print("edge multiplicities > 1:", {e: m for e, m in Counter(map(tuple, map(sorted, G.edges()))).items() if m > 1})

# reproducibility: compare the edge lists directly
same = configuration_model(degrees, seed=10)
other = configuration_model(degrees, seed=11)
print("same seed, same graph:", list(G.edges()) == list(same.edges()))
print("different seed, same graph:", list(G.edges()) == list(other.edges()))

Now the bad inputs. An odd sum can't be paired up, and negative or non-integer degrees make no sense, so these should all raise a ValueError.

In [ ]:
# odd sum, negative degree, non-integer degree
for bad in ([1, 2], [1, -1], [1.0, 1]):
    try:
        configuration_model(bad)
    except ValueError as e:
        print(bad, "->", e)

# empty sequence and all-zero sequence should work and give no edges
print(configuration_model([]).number_of_nodes(), configuration_model([0, 0]).number_of_edges())

Finally, we compare against `nx.configuration_model`. We can't compare single graphs since both are random (and the two use different random number generators, so the same seed won't give the same graph), so instead we sample 500 graphs from each, using seeds 10 through 509. Both should hit the degree sequence exactly every time (the assert inside `run_model`), and the average number of self-loops and duplicate edges should be close if the pairing is uniform.

In [ ]:
def run_model(make, degrees, n_samples=500, seed=10):
    # sample n_samples graphs from a generator (sample i uses seed + i), checking
    # the degree sequence every time and recording self-loops and redundant
    # (duplicate) edges
    loops, extra = [], []
    for i in range(n_samples):
        H = make(degrees, seed=seed + i)
        assert [H.degree(i) for i in range(len(degrees))] == degrees
        loops.append(nx.number_of_selfloops(H))
        # nx.Graph merges parallel edges but keeps self-loops, so this counts
        # only the extra copies of an edge
        extra.append(H.number_of_edges() - nx.Graph(H).number_of_edges())
    return np.mean(loops), np.mean(extra)

mine = run_model(configuration_model, degrees)
ref = run_model(nx.configuration_model, degrees)
print(f"mine:     mean self-loops {mine[0]:.3f}, mean duplicate edges {mine[1]:.3f}")
print(f"networkx: mean self-loops {ref[0]:.3f}, mean duplicate edges {ref[1]:.3f}")